# Hotel Booking Analysis - Detailed Insights

## Comprehensive Analysis Report with Visualizations and Key Findings

## Section 1: Data Loading and Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Load data
df = pd.read_csv('hotel_bookings.csv')
print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumn Names: {df.columns.tolist()}")

In [ ]:
# Data Overview
print("\n=== DATASET OVERVIEW ===")
print(f"Total Records: {len(df):,}")
print(f"Total Columns: {len(df.columns)}")
print(f"Date Range: {df['arrival_date_year'].min():.0f} - {df['arrival_date_year'].max():.0f}")
print(f"\nData Types:")
print(df.dtypes)
print(f"\nMissing Values: {df.isnull().sum().sum()}")
print(f"Duplicate Rows: {df.duplicated().sum():,}")

## Section 2: Booking Volume & Cancellation Analysis

In [ ]:
# Cancellation Overview
print("\n=== BOOKING & CANCELLATION STATISTICS ===")
total_bookings = len(df)
canceled = df['is_canceled'].sum()
completed = total_bookings - canceled
cancel_rate = (canceled / total_bookings) * 100

print(f"Total Bookings: {total_bookings:,}")
print(f"Completed: {completed:,} ({100-cancel_rate:.1f}%)")
print(f"Canceled: {int(canceled):,} ({cancel_rate:.1f}%)")
print(f"\n⚠️  Cancellation Rate: {cancel_rate:.1f}%")

if cancel_rate > 30:
    print("   INSIGHT: Very high cancellation rate - critical attention needed!")
elif cancel_rate > 20:
    print("   INSIGHT: Moderate-to-high cancellations - implement mitigation strategies")
else:
    print("   INSIGHT: Acceptable cancellation rate for hospitality industry")

In [ ]:
# Visualization: Cancellation Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
cancel_counts = df['is_canceled'].value_counts()
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(['Completed', 'Canceled'], [cancel_counts[0], cancel_counts[1]], color=colors, edgecolor='black', linewidth=1.5)
axes[0].set_title('Booking Status Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Bookings', fontsize=11)
for i, v in enumerate(cancel_counts):
    axes[0].text(i, v + 5000, f'{int(v):,}\n({v/total_bookings*100:.1f}%)', ha='center', fontweight='bold')

# Pie chart
axes[1].pie([cancel_counts[0], cancel_counts[1]], labels=['Completed', 'Canceled'], autopct='%1.1f%%', 
             colors=colors, startangle=90, textprops={'fontsize': 11, 'weight': 'bold'})
axes[1].set_title('Booking Status Breakdown', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('01_cancellation_overview.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Visualization saved: 01_cancellation_overview.png")

## Section 3: Hotel Type & Customer Segments

In [ ]:
# Hotel Type Analysis
print("\n=== HOTEL TYPE DISTRIBUTION ===")
hotel_dist = df['hotel'].value_counts()
print(hotel_dist)
for hotel, count in hotel_dist.items():
    pct = (count / len(df)) * 100
    print(f"{hotel}: {count:,} ({pct:.1f}%)")

# Cancellation by hotel type
print("\nCancellation Rate by Hotel Type:")
cancel_by_hotel = df.groupby('hotel')['is_canceled'].agg(['sum', 'count', 'mean'])
cancel_by_hotel['cancel_rate'] = (cancel_by_hotel['mean'] * 100).round(1)
print(cancel_by_hotel[['cancel_rate']])

In [ ]:
# Market Segment Analysis
print("\n=== MARKET SEGMENT DISTRIBUTION ===")
segment_dist = df['market_segment'].value_counts()
for segment, count in segment_dist.items():
    pct = (count / len(df)) * 100
    print(f"{segment}: {count:,} ({pct:.1f}%)")

# Cancellation by market segment
print("\nCancellation Rate by Market Segment:")
cancel_by_segment = df.groupby('market_segment')['is_canceled'].mean() * 100
print(cancel_by_segment.round(1))

# Identify highest risk segment
highest_cancel_segment = cancel_by_segment.idxmax()
highest_cancel_rate = cancel_by_segment.max()
print(f"\n⚠️  HIGHEST CANCELLATION RISK: {highest_cancel_segment} ({highest_cancel_rate:.1f}%)")

In [ ]:
# Visualization: Hotel Type and Market Segments
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Hotel type
hotel_data = df['hotel'].value_counts()
axes[0].barh(hotel_data.index, hotel_data.values, color=['#3498db', '#e74c3c'], edgecolor='black')
axes[0].set_title('Bookings by Hotel Type', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Bookings', fontsize=11)
for i, v in enumerate(hotel_data.values):
    axes[0].text(v + 1000, i, f'{int(v):,} ({v/len(df)*100:.1f}%)', va='center', fontweight='bold')

# Market segment
segment_data = df['market_segment'].value_counts()
colors_seg = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c', '#9b59b6']
axes[1].bar(range(len(segment_data)), segment_data.values, color=colors_seg[:len(segment_data)], edgecolor='black')
axes[1].set_xticks(range(len(segment_data)))
axes[1].set_xticklabels(segment_data.index, rotation=45, ha='right')
axes[1].set_title('Bookings by Market Segment', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Number of Bookings', fontsize=11)
for i, v in enumerate(segment_data.values):
    axes[1].text(i, v + 1000, f'{int(v):,}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('02_hotel_and_segments.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Visualization saved: 02_hotel_and_segments.png")

## Section 4: Seasonal Patterns & Monthly Trends

In [ ]:
# Monthly Analysis
print("\n=== SEASONAL & MONTHLY ANALYSIS ===")
month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
                'July', 'August', 'September', 'October', 'November', 'December']

monthly_bookings = df['arrival_date_month'].value_counts().reindex(month_order)
print("\nMonthly Booking Distribution:")
for month, count in monthly_bookings.items():
    pct = (count / len(df)) * 100
    bar = '█' * int(pct / 2)
    print(f"{month:12s}: {count:6,.0f} ({pct:5.1f}%) {bar}")

# Peak and low seasons
peak_month = monthly_bookings.idxmax()
low_month = monthly_bookings.idxmin()
seasonality_ratio = monthly_bookings.max() / monthly_bookings.min()

print(f"\nSEASONALITY INSIGHTS:")
print(f"📈 Peak Season: {peak_month} ({monthly_bookings[peak_month]:,.0f} bookings)")
print(f"📉 Low Season: {low_month} ({monthly_bookings[low_month]:,.0f} bookings)")
print(f"📊 Seasonality Ratio: {seasonality_ratio:.2f}x (Peak/Low)")

if seasonality_ratio > 2:
    print("   ⚠️  CRITICAL: Extreme seasonality - need dynamic pricing & capacity planning")

In [ ]:
# Visualization: Monthly Trends
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Bookings by month
monthly_bookings_sorted = df['arrival_date_month'].value_counts().reindex(month_order)
axes[0].plot(range(len(monthly_bookings_sorted)), monthly_bookings_sorted.values, 
             marker='o', linewidth=2.5, markersize=8, color='#3498db')
axes[0].fill_between(range(len(monthly_bookings_sorted)), monthly_bookings_sorted.values, 
                      alpha=0.3, color='#3498db')
axes[0].set_xticks(range(len(month_order)))
axes[0].set_xticklabels([m[:3] for m in month_order], fontsize=10)
axes[0].set_title('Monthly Booking Trends', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Bookings', fontsize=11)
axes[0].grid(True, alpha=0.3)

# Cancellation rate by month
cancel_by_month = df.groupby('arrival_date_month')['is_canceled'].mean().reindex(month_order) * 100
axes[1].bar(range(len(cancel_by_month)), cancel_by_month.values, 
             color=['#e74c3c' if x > 30 else '#f39c12' if x > 20 else '#2ecc71' for x in cancel_by_month.values],
             edgecolor='black')
axes[1].axhline(y=cancel_by_month.mean(), color='red', linestyle='--', linewidth=2, label=f'Average: {cancel_by_month.mean():.1f}%')
axes[1].set_xticks(range(len(month_order)))
axes[1].set_xticklabels([m[:3] for m in month_order], fontsize=10)
axes[1].set_title('Cancellation Rate by Month', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Cancellation Rate (%)', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('03_seasonal_trends.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Visualization saved: 03_seasonal_trends.png")

## Section 5: Lead Time & Advance Booking Analysis

In [ ]:
# Lead Time Analysis
print("\n=== LEAD TIME ANALYSIS ===")
print(f"Average Lead Time: {df['lead_time'].mean():.1f} days")
print(f"Median Lead Time: {df['lead_time'].median():.1f} days")
print(f"Std Dev: {df['lead_time'].std():.1f} days")
print(f"Range: {df['lead_time'].min():.0f} - {df['lead_time'].max():.0f} days")

# Lead time categories
df['lead_time_category'] = pd.cut(df['lead_time'], 
                                   bins=[0, 0, 7, 14, 30, 90, 365],
                                   labels=['Same Day', '1-7 days', '8-14 days', '15-30 days', '31-90 days', '>90 days'])

print("\nBooking Lead Time Distribution:")
lead_dist = df['lead_time_category'].value_counts().sort_index()
for category, count in lead_dist.items():
    pct = (count / len(df)) * 100
    cancel_rate = df[df['lead_time_category'] == category]['is_canceled'].mean() * 100
    print(f"{category:15s}: {count:6,.0f} ({pct:5.1f}%) - Cancellation Rate: {cancel_rate:5.1f}%")

print("\n💡 INSIGHT: Longer lead times are associated with higher cancellation rates")

In [ ]:
# Visualization: Lead Time Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Lead time distribution
lead_dist_sorted = df['lead_time_category'].value_counts().sort_index()
axes[0].bar(range(len(lead_dist_sorted)), lead_dist_sorted.values, 
             color='#9b59b6', edgecolor='black', alpha=0.7)
axes[0].set_xticks(range(len(lead_dist_sorted)))
axes[0].set_xticklabels(lead_dist_sorted.index, rotation=45, ha='right')
axes[0].set_title('Distribution of Booking Lead Times', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Bookings', fontsize=11)
for i, v in enumerate(lead_dist_sorted.values):
    axes[0].text(i, v + 1000, f'{int(v):,}', ha='center', fontweight='bold', fontsize=9)

# Cancellation by lead time
cancel_by_lead = df.groupby('lead_time_category')['is_canceled'].mean() * 100
cancel_by_lead_sorted = cancel_by_lead.sort_index()
colors_lead = ['#2ecc71' if x < 20 else '#f39c12' if x < 30 else '#e74c3c' for x in cancel_by_lead_sorted.values]
axes[1].bar(range(len(cancel_by_lead_sorted)), cancel_by_lead_sorted.values, 
             color=colors_lead, edgecolor='black')
axes[1].axhline(y=df['is_canceled'].mean()*100, color='red', linestyle='--', linewidth=2, label='Overall Avg')
axes[1].set_xticks(range(len(cancel_by_lead_sorted)))
axes[1].set_xticklabels(cancel_by_lead_sorted.index, rotation=45, ha='right')
axes[1].set_title('Cancellation Rate by Lead Time', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Cancellation Rate (%)', fontsize=11)
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('04_lead_time_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Visualization saved: 04_lead_time_analysis.png")

## Section 6: Pricing & Revenue Analysis

In [ ]:
# ADR (Average Daily Rate) Analysis
print("\n=== PRICING & REVENUE ANALYSIS ===")
print(f"\nAverage Daily Rate (ADR) Statistics:")
print(f"  Mean: ${df['adr'].mean():.2f}")
print(f"  Median: ${df['adr'].median():.2f}")
print(f"  Std Dev: ${df['adr'].std():.2f}")
print(f"  Min: ${df['adr'].min():.2f}")
print(f"  Max: ${df['adr'].max():.2f}")

# ADR by hotel type
print(f"\nAverage Daily Rate by Hotel Type:")
adr_by_hotel = df.groupby('hotel')['adr'].agg(['mean', 'median', 'count'])
adr_by_hotel.columns = ['Mean ADR', 'Median ADR', 'Bookings']
print(adr_by_hotel.round(2))

# Revenue Analysis (Completed bookings only)
df_completed = df[df['is_canceled'] == 0]
df_completed['total_stay'] = df_completed['stays_in_weekend_nights'] + df_completed['stays_in_week_nights']
df_completed['revenue'] = df_completed['adr'] * df_completed['total_stay']

total_revenue = df_completed['revenue'].sum()
avg_revenue_per_booking = df_completed['revenue'].mean()

print(f"\nRevenue Metrics (Completed Bookings):")
print(f"  Total Revenue: ${total_revenue:,.2f}")
print(f"  Average Revenue per Booking: ${avg_revenue_per_booking:.2f}")
print(f"  Total Completed Bookings: {len(df_completed):,}")

In [ ]:
# Visualization: Pricing Analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ADR distribution
axes[0, 0].hist(df['adr'], bins=50, color='#3498db', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df['adr'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: ${df["adr"].mean():.2f}')
axes[0, 0].axvline(df['adr'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: ${df["adr"].median():.2f}')
axes[0, 0].set_title('Distribution of Average Daily Rates', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('ADR ($)', fontsize=10)
axes[0, 0].set_ylabel('Frequency', fontsize=10)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3, axis='y')

# ADR by hotel type
adr_by_hotel_data = df.groupby('hotel')['adr'].mean()
axes[0, 1].bar(adr_by_hotel_data.index, adr_by_hotel_data.values, color=['#3498db', '#e74c3c'], edgecolor='black')
axes[0, 1].set_title('Average Daily Rate by Hotel Type', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('ADR ($)', fontsize=10)
for i, v in enumerate(adr_by_hotel_data.values):
    axes[0, 1].text(i, v + 5, f'${v:.2f}', ha='center', fontweight='bold')

# ADR by market segment
adr_by_segment = df.groupby('market_segment')['adr'].mean().sort_values(ascending=False)
axes[1, 0].barh(adr_by_segment.index, adr_by_segment.values, color='#2ecc71', edgecolor='black')
axes[1, 0].set_title('Average Daily Rate by Market Segment', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('ADR ($)', fontsize=10)
for i, v in enumerate(adr_by_segment.values):
    axes[1, 0].text(v + 2, i, f'${v:.2f}', va='center', fontweight='bold')

# Length of stay vs ADR (scatter)
df_scatter = df[(df['adr'] < df['adr'].quantile(0.95))].copy()  # Remove outliers for better visualization
df_scatter['total_stay'] = df_scatter['stays_in_weekend_nights'] + df_scatter['stays_in_week_nights']
axes[1, 1].scatter(df_scatter['total_stay'], df_scatter['adr'], alpha=0.3, s=10, color='#9b59b6')
axes[1, 1].set_title('Length of Stay vs Average Daily Rate', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Total Stay (nights)', fontsize=10)
axes[1, 1].set_ylabel('ADR ($)', fontsize=10)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('05_pricing_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Visualization saved: 05_pricing_analysis.png")

## Section 7: Guest Composition & Length of Stay

In [ ]:
# Guest Analysis
print("\n=== GUEST COMPOSITION ANALYSIS ===")
df['total_guests'] = df['adults'] + df['children']
df['total_stay'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

print(f"\nGuest Statistics:")
print(f"  Average Adults per Booking: {df['adults'].mean():.2f}")
print(f"  Average Children per Booking: {df['children'].mean():.2f}")
print(f"  Average Total Guests: {df['total_guests'].mean():.2f}")
print(f"  Max Guests in Single Booking: {df['total_guests'].max():.0f}")

print(f"\nLength of Stay Statistics:")
print(f"  Average Total Stay: {df['total_stay'].mean():.2f} nights")
print(f"  Median Total Stay: {df['total_stay'].median():.2f} nights")
print(f"  Max Stay: {df['total_stay'].max():.0f} nights")

# Stay categories
df['stay_category'] = pd.cut(df['total_stay'], 
                              bins=[0, 3, 7, 14, 30, 365],
                              labels=['1-3 nights', '4-7 nights', '8-14 nights', '15-30 nights', '>30 nights'])

print(f"\nBooking Duration Distribution:")
stay_dist = df['stay_category'].value_counts().sort_index()
for category, count in stay_dist.items():
    pct = (count / len(df)) * 100
    avg_adr = df[df['stay_category'] == category]['adr'].mean()
    print(f"{category:15s}: {count:6,.0f} ({pct:5.1f}%) - Avg ADR: ${avg_adr:7.2f}")

In [ ]:
# Visualization: Guest & Stay Analysis
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Guest composition
axes[0, 0].bar(['Adults', 'Children'], 
                [df['adults'].sum(), df['children'].sum()], 
                color=['#3498db', '#e74c3c'], edgecolor='black')
axes[0, 0].set_title('Total Guest Composition', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Total Count', fontsize=10)
axes[0, 0].ticklabel_format(style='plain', axis='y')

# Stay categories
stay_dist_sorted = df['stay_category'].value_counts().sort_index()
axes[0, 1].bar(range(len(stay_dist_sorted)), stay_dist_sorted.values, 
                color='#2ecc71', edgecolor='black', alpha=0.7)
axes[0, 1].set_xticks(range(len(stay_dist_sorted)))
axes[0, 1].set_xticklabels(stay_dist_sorted.index, rotation=45, ha='right')
axes[0, 1].set_title('Distribution of Length of Stay', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Number of Bookings', fontsize=10)

# Stay distribution histogram
axes[1, 0].hist(df['total_stay'], bins=50, color='#9b59b6', edgecolor='black', alpha=0.7)
axes[1, 0].axvline(df['total_stay'].mean(), color='red', linestyle='--', linewidth=2, 
                    label=f'Mean: {df["total_stay"].mean():.1f} nights')
axes[1, 0].set_title('Total Stay Distribution', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Number of Nights', fontsize=10)
axes[1, 0].set_ylabel('Frequency', fontsize=10)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Total guests distribution
axes[1, 1].hist(df['total_guests'], bins=range(0, df['total_guests'].max()+2), 
                 color='#f39c12', edgecolor='black', alpha=0.7)
axes[1, 1].set_title('Distribution of Total Guests per Booking', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Number of Guests', fontsize=10)
axes[1, 1].set_ylabel('Frequency', fontsize=10)
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('06_guest_analysis.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ Visualization saved: 06_guest_analysis.png")

## Section 8: Key Performance Indicators (KPIs) Summary

In [ ]:
# KPI Summary
print("\n" + "="*80)
print("KEY PERFORMANCE INDICATORS (KPIs) SUMMARY")
print("="*80)

kpis = {
    'Total Bookings': f"{len(df):,}",
    'Completed Bookings': f"{len(df[df['is_canceled']==0]):,}",
    'Cancellation Rate': f"{df['is_canceled'].mean()*100:.1f}%",
    'Cancellation Count': f"{int(df['is_canceled'].sum()):,}",
    'Average Daily Rate': f"${df['adr'].mean():.2f}",
    'Total Revenue': f"${total_revenue:,.2f}",
    'Avg Revenue per Booking': f"${avg_revenue_per_booking:.2f}",
    'Avg Length of Stay': f"{df['total_stay'].mean():.1f} nights",
    'Avg Guests per Booking': f"{df['total_guests'].mean():.1f}",
    'Avg Lead Time': f"{df['lead_time'].mean():.0f} days",
    'Peak Season Month': peak_month,
    'Seasonality Ratio': f"{seasonality_ratio:.2f}x"
}

for kpi, value in kpis.items():
    print(f"{kpi:.<40} {value:>35}")

print("="*80)

## Section 9: Critical Insights & Findings

In [ ]:
print("\n" + "="*80)
print("CRITICAL INSIGHTS & FINDINGS")
print("="*80)

insights_list = []

# Insight 1: Cancellation
cancel_rate = df['is_canceled'].mean() * 100
if cancel_rate > 30:
    insights_list.append(f"🔴 CRITICAL: {cancel_rate:.1f}% cancellation rate - Implement immediate mitigation strategies")
elif cancel_rate > 20:
    insights_list.append(f"🟠 WARNING: {cancel_rate:.1f}% cancellation rate - Above industry average (15-20%)")
else:
    insights_list.append(f"🟢 GOOD: {cancel_rate:.1f}% cancellation rate - Within acceptable range")

# Insight 2: Seasonality
if seasonality_ratio > 3:
    insights_list.append(f"🔴 EXTREME SEASONALITY: {seasonality_ratio:.2f}x variance - Requires aggressive dynamic pricing")
elif seasonality_ratio > 2:
    insights_list.append(f"🟠 HIGH SEASONALITY: {seasonality_ratio:.2f}x variance - Peak months significantly outperform low months")

# Insight 3: Lead Time Risk
long_lead_cancel = df[df['lead_time'] > 90]['is_canceled'].mean() * 100
short_lead_cancel = df[df['lead_time'] <= 7]['is_canceled'].mean() * 100
if long_lead_cancel > short_lead_cancel * 1.5:
    insights_list.append(f"⚠️  LEAD TIME RISK: Long-lead bookings cancel at {long_lead_cancel:.1f}% vs {short_lead_cancel:.1f}% for short-lead")

# Insight 4: Highest Risk Segment
cancel_by_segment = df.groupby('market_segment')['is_canceled'].mean() * 100
highest_risk = cancel_by_segment.idxmax()
highest_risk_rate = cancel_by_segment.max()
insights_list.append(f"⚠️  HIGH-RISK SEGMENT: '{highest_risk}' has {highest_risk_rate:.1f}% cancellation rate")

# Insight 5: Pricing
adr_variation = df['adr'].std() / df['adr'].mean()
if adr_variation > 1:
    insights_list.append(f"💰 HIGH PRICE VARIATION: ADR ranges from ${df['adr'].min():.0f} to ${df['adr'].max():.0f}")

# Insight 6: Revenue Impact
lost_revenue = (df[df['is_canceled'] == 1]['adr'] * (df[df['is_canceled'] == 1]['stays_in_weekend_nights'] + df[df['is_canceled'] == 1]['stays_in_week_nights'])).sum()
insights_list.append(f"💸 LOST REVENUE: ${lost_revenue:,.2f} from cancellations ({lost_revenue/(total_revenue+lost_revenue)*100:.1f}% of potential revenue)")

# Insight 7: Hotel Type Performance
hotel_cancel_rates = df.groupby('hotel')['is_canceled'].mean() * 100
worst_hotel = hotel_cancel_rates.idxmax()
worst_rate = hotel_cancel_rates.max()
insights_list.append(f"🏨 HOTEL PERFORMANCE: {worst_hotel} has higher cancellation ({worst_rate:.1f}%) than average")

for i, insight in enumerate(insights_list, 1):
    print(f"\n{i}. {insight}")

print("\n" + "="*80)

## Section 10: Strategic Recommendations

In [ ]:
print("\n" + "="*80)
print("STRATEGIC RECOMMENDATIONS")
print("="*80)

recommendations = [
    {
        'title': 'CANCELLATION RISK MITIGATION',
        'actions': [
            '• Implement non-refundable rate options with 10-15% discount',
            '• Add prepayment requirements for high-risk segments',
            f'• Focus on {highest_risk} segment with special retention offers',
            '• Create early-bird booking incentives to lock in reservations'
        ]
    },
    {
        'title': 'SEASONALITY MANAGEMENT',
        'actions': [
            '• Implement dynamic pricing: increase rates in peak months',
            f'• Create promotional campaigns for low season ({low_month})',
            '• Bundle room packages with experiences/attractions',
            '• Develop group packages for corporate bookings in low season'
        ]
    },
    {
        'title': 'LEAD TIME OPTIMIZATION',
        'actions': [
            '• Create incentives for advance bookings (3+ months ahead)',
            '• Offer last-minute deals to fill cancellations',
            '• Implement graduated cancellation policies based on lead time',
            '• Develop forecasting model to predict cancellations'
        ]
    },
    {
        'title': 'REVENUE OPTIMIZATION',
        'actions': [
            f'• Focus marketing on high-ADR segments',
            '• Upsell ancillary services to increase ADR',
            '• Create tiered pricing based on stay length',
            f'• Target recovery of ${lost_revenue:,.0f} lost revenue'
        ]
    },
    {
        'title': 'CUSTOMER SEGMENT STRATEGY',
        'actions': [
            '• Develop segment-specific retention programs',
            f'• Special attention to {highest_risk} (highest cancellation)',
            '• Create loyalty tiers for frequent bookers',
            '• Implement personalized communications per segment'
        ]
    },
    {
        'title': 'OPERATIONAL IMPROVEMENTS',
        'actions': [
            '• Monitor KPIs weekly and adjust tactics',
            '• Train staff on cancellation prevention',
            '• Implement automated confirmation/reminder systems',
            '• Set up A/B testing for pricing and offers'
        ]
    }
]

for rec in recommendations:
    print(f"\n📋 {rec['title']}")
    for action in rec['actions']:
        print(f"   {action}")

print("\n" + "="*80)

## Section 11: Priority Action Items

In [ ]:
print("\n" + "="*80)
print("PRIORITY ACTION ITEMS (Next 30 Days)")
print("="*80)

priority_actions = [
    {
        'priority': '🔴 URGENT',
        'action': 'Analyze cancellation patterns',
        'timeframe': 'Week 1',
        'owner': 'Analytics Team',
        'impact': f'Prevent ${lost_revenue/4:,.0f} monthly revenue loss'
    },
    {
        'priority': '🔴 URGENT',
        'action': f'Launch retention campaign for {highest_risk} segment',
        'timeframe': 'Week 1-2',
        'owner': 'Marketing Team',
        'impact': f'Reduce {highest_risk} cancellations by 5-10%'
    },
    {
        'priority': '🟠 HIGH',
        'action': 'Implement dynamic pricing for peak season',
        'timeframe': 'Week 2-3',
        'owner': 'Revenue Management',
        'impact': f'Increase revenue by 10-15% in peak months'
    },
    {
        'priority': '🟠 HIGH',
        'action': 'Launch low-season promotional campaigns',
        'timeframe': 'Week 2-4',
        'owner': 'Marketing Team',
        'impact': f'Boost {low_month} bookings by 20%'
    },
    {
        'priority': '🟡 MEDIUM',
        'action': 'Set up KPI tracking dashboard',
        'timeframe': 'Week 3-4',
        'owner': 'Analytics Team',
        'impact': 'Real-time monitoring of key metrics'
    }
]

for item in priority_actions:
    print(f"\n{item['priority']} | {item['action']}")
    print(f"    Timeframe: {item['timeframe']} | Owner: {item['owner']}")
    print(f"    Expected Impact: {item['impact']}")

print("\n" + "="*80)
print("END OF DETAILED ANALYSIS REPORT")
print("="*80)